## Semantic search
Semantic search finds information based on **meaning**, not just exact keyword matches. In LLM applications, text is converted into **embeddings** so similar ideas are placed close together in vector space.

In a **RAG** workflow, semantic search is used to retrieve the most relevant chunks of documents for a user’s question. Those retrieved chunks are then passed to the LLM as context, helping it generate answers that are more accurate, grounded, and specific to the source material.

### Langchain `PyPDFLoader`

- **Simple PDF ingestion**: Quickly loads PDF files into LangChain `Document` objects.
- **Page-aware parsing**: Preserves page-level structure and metadata, useful for citations and tracing sources.
- **RAG-ready format**: Output works directly with text splitters, embeddings, and vector stores.
- **Less boilerplate**: Reduces custom PDF parsing code so you can focus on retrieval and QA logic.
- **Works well in pipelines**: Integrates cleanly with common LangChain components for end-to-end semantic search workflows.

In [1]:
%run initialize_environment.py
from langchain_community.document_loaders import PyPDFLoader

Environment initializing completed successfully.


In [2]:
loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

docs = loader.load()
for doc in docs:
    pprint(doc.metadata)

incorrect startxref pointer(1)
parsing for Object Streams


{'author': '(anonymous)',
 'creationdate': '2025-11-20T23:23:16+00:00',
 'creator': '(unspecified)',
 'keywords': '',
 'moddate': '2025-11-20T23:23:16+00:00',
 'page': 0,
 'page_label': '1',
 'producer': 'ReportLab PDF Library - www.reportlab.com',
 'source': 'resources/acmecorp-employee-handbook.pdf',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'total_pages': 1,
 'trapped': '/False'}


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

3


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [5]:
embeddings = create_azure_embedding()

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [7]:
ids = vector_store.add_documents(documents=all_splits)

In [8]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

## RAG Agent

In [9]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=create_azure_llm(),
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [11]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='How many days of vacation does an employee get in their first year?', additional_kwargs={}, response_metadata={}, id='cf161482-6626-4703-bcc7-590e93c1a05a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 73, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DPNlGk5w9sV1Aj5J8M8UmjPZecZ3w', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity':